In [29]:
from importlib import import_module
import torch
import torch.nn as nn
import math

print(torch.cuda.is_available())
print("torch version: ", torch.version.cuda)


False
torch version:  12.4


# 1.手撕注意力机制

## 1.1.构造输入数据

In [30]:
keys_embs = {
    "金毛": [0.2, 0.4, 0.8, -0.1],
    "西瓜": [-3.7, -4.2, 1.1, 2.1],
    "丁师兄": [-2.4, 0.1, -0.3, 4.5]
}

queries_embs = {
    "动物": [0.11, 0.19, 0.45, -0.06],
    "水果": [-1.7, -1.4, 0.7, 0.1]
}

query1 = torch.FloatTensor([queries_embs["动物"]])
print("query: ", query1, query1.shape)

key = torch.FloatTensor(list(keys_embs.values()))
print("key: ", key, key.shape)

value = torch.FloatTensor(list(keys_embs.values()))
print("value: ", value, value.shape)

query:  tensor([[ 0.1100,  0.1900,  0.4500, -0.0600]]) torch.Size([1, 4])
key:  tensor([[ 0.2000,  0.4000,  0.8000, -0.1000],
        [-3.7000, -4.2000,  1.1000,  2.1000],
        [-2.4000,  0.1000, -0.3000,  4.5000]]) torch.Size([3, 4])
value:  tensor([[ 0.2000,  0.4000,  0.8000, -0.1000],
        [-3.7000, -4.2000,  1.1000,  2.1000],
        [-2.4000,  0.1000, -0.3000,  4.5000]]) torch.Size([3, 4])


## 1.2. 注意力机制实现

In [31]:
def attention(query, key, value):
    """
    注意力机制实现
    """
    d_k = query.size(-1)

    score = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)
    # 为啥是dim=-1因为-1是按照最后一个维度进行softmax，二维里面是列维度移动，也就是一行一行的计算
    attention = torch.softmax(score, dim=-1)

    output = torch.matmul(attention, value)
    return score, attention, output


## 1.3. 开始计算动物

In [32]:
unm_score, att_score, att = attention(query1, key, value)
print(unm_score)
print(att_score)
print(att)

tensor([[ 0.2320, -0.4180, -0.3250]])
tensor([[0.4773, 0.2492, 0.2735]])
tensor([[-1.4829, -0.8283,  0.5739,  1.7062]])


tensor([[ 0.2320, -0.4180, -0.3250]])可以看到金毛和动物的相关性最高是正的0.2320\
tensor([[0.4773, 0.2492, 0.2735]]) 可以看到动物和金毛的相关性最高是正的0.4773\
tensor([[-1.4829, -0.8283,  0.5739,  1.7062]]) \
att = 0.4773 × [0.2, 0.4, 0.8, -0.1]    (金毛)\
    + 0.2492 × [-3.7, -4.2, 1.1, 2.1]   (西瓜)\
    + 0.2735 × [-2.4, 0.1, -0.3, 4.5]   (丁师兄)

## 1.4. 开始计算水果

In [33]:
query2 = torch.FloatTensor([queries_embs["水果"]])
unm_scores, att_score, output = attention(query2, key, value)
print("unm_scores: ", unm_scores)
print("att_score: ", att_score)
print("output: ", output)

unm_scores:  tensor([[-0.1750,  6.5750,  2.0900]])
att_score:  tensor([[0.0012, 0.9877, 0.0111]])
output:  tensor([[-3.6810, -4.1468,  1.0841,  2.1242]])


其实能算出动物和金毛的相关性是因为基础的向量里面金毛的向量和动物的向量的前3个维度都是高度相同的

## 1.5. 开始计算全部的

In [34]:
query3 = torch.FloatTensor(list(queries_embs.values()))
unm_scores, att_score, output = attention(query3, key, value)
print("unm_scores: ", unm_scores)
print("att_score: ", att_score)
print("output: ", output)

unm_scores:  tensor([[ 0.2320, -0.4180, -0.3250],
        [-0.1750,  6.5750,  2.0900]])
att_score:  tensor([[0.4773, 0.2492, 0.2735],
        [0.0012, 0.9877, 0.0111]])
output:  tensor([[-1.4829, -0.8283,  0.5739,  1.7062],
        [-3.6810, -4.1468,  1.0841,  2.1242]])
